<a href="https://colab.research.google.com/github/liminalvoid/nlp/blob/main/sem_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Семинар 4. Word2Vec и простой retrieval

## Настройка, установка и импорт необходимых библиотек

In [54]:
!pip install -q gensim faiss-cpu

import re
import zipfile
import os
import tempfile

import numpy as np
import gensim
import gensim.downloader as api
import requests
import faiss

from collections import Counter
from typing import List

from datasets import load_dataset
from gensim.models import Word2Vec
from gensim.models import KeyedVectors
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

## Загрузка модели эмбеддингов для русского языка, анализ требуемого препроцессинга

В качестве модели эмбеддингов для русского языка используется [ruscorpora_upos_cbow_300_20_2019](https://rusvectores.org/en/models/#:~:text=ruscorpora%5Fupos%5Fcbow%5F300%5F20%5F2019). Скачаем архив с моделью.

In [ ]:
MODEL_PATH = "./model.zip"
MODEL_URL = "https://vectors.nlpl.eu/repository/20/180.zip"

r = requests.get(MODEL_URL)

with open(MODEL_PATH, "wb") as f:
    f.write(r.content)


Далее загрузим модель из скаченного архива.

In [43]:
model = None

with zipfile.ZipFile(MODEL_PATH, "r") as archive:
    with tempfile.TemporaryDirectory() as tmp:
        archive.extract("model.txt", path=tmp)
        txt_path = os.path.join(tmp, "model.txt")
        model = gensim.models.KeyedVectors.load_word2vec_format(
            txt_path,
            binary=False,
            unicode_errors="replace",
        )

Для того, чтобы понять какой препроцессинг требуется для данной модели, выведем первые 20 слов из её словаря. В контексте требуемой моделью предобработки можно отметить наличие POS-тегов и то, что все слова находятся в нижнем регистре.

In [44]:
list(model.key_to_index.keys())[:20]

['так_ADV',
 'быть_VERB',
 'мочь_VERB',
 'год_NOUN',
 'человек_NOUN',
 'xxxxxx_NUM',
 'сказать_VERB',
 'еще_ADV',
 'один_NUM',
 'говорить_VERB',
 'уже_ADV',
 'другой_ADJ',
 'время_NOUN',
 'xxxxxxxx_NUM',
 'знать_VERB',
 'сам_ADJ',
 'самый_ADJ',
 'делать_VERB',
 'дело_NOUN',
 'день_NOUN']

Далее возьмем 20 слов (по 5 на каждый домен) и посмотрим 10 ближайших соседей для каждого из них.

Ближайшими соседями некоторых слов являются их производные (например, котенок и котенка). Это говорит о том, что препроцессинг корпуса не предполагает лемматизацию.

Также можно отметить странность в тегах для некоторых слов: например, для слова "волчий" стоит тег "NOUN" (существительное), когда логичнее было бы использовать тег "ADJ" (прилагательное).

In [45]:
def print_neighbors(wv, word, top_n=10):
    if wv is None:
        print("Нет загруженных эмбеддингов (wv = None)")

        return

    if word not in wv:
        print(f"\"{word}\" нет в словаре модели")

        return

    neighbors = wv.most_similar(word, topn=top_n)
    print(word, "->", [w for w, _ in neighbors])


words = {
    "животные": ["кошка", "собака", "лошадь", "медведь", "волк"],
    "еда": ["хлеб", "молоко", "мясо", "сахар", "масло"],
    "семья": ["мать", "отец", "брат", "сестра", "дочь"],
    "погода": ["дождь", "снег", "ветер", "солнце", "мороз"],
}

for domain in words:
    print("Домен:", domain)

    for word in words[domain]:
        print_neighbors(model, word + "_NOUN")

Домен: животные
кошка_NOUN -> ['кот_NOUN', 'котенок_NOUN', 'котенка_NOUN', 'котенок_VERB', 'собака_NOUN', 'щенок_NOUN', 'собачка_NOUN', 'кошка_PROPN', 'дворняжка_NOUN', 'крыса_NOUN']
собака_NOUN -> ['пес_NOUN', 'дворняжка_NOUN', 'дворняга_NOUN', 'легавый_ADJ', 'кошка_NOUN', 'собачонка_NOUN', 'овчарка_NOUN', 'щенок_NOUN', 'гончий_ADJ', 'собачка_NOUN']
лошадь_NOUN -> ['конь_NOUN', 'лошадь_PROPN', 'жеребец_NOUN', 'лошадь_ADV', 'кобыла_NOUN', 'повозка_NOUN', 'лошадка_NOUN', 'телега_NOUN', 'иноходец_NOUN', 'мул_NOUN']
медведь_NOUN -> ['медведь_PROPN', 'медведь_VERB', 'зверь_NOUN', 'волк_NOUN', 'медведица_NOUN', 'барсук_NOUN', 'тигр_NOUN', 'заяц_NOUN', 'кабан_NOUN', 'медвежонок_NOUN']
волк_NOUN -> ['медведь_NOUN', 'зверь_NOUN', 'волк_PROPN', 'заяц_NOUN', 'волчица_NOUN', 'лисица_NOUN', 'лиса_NOUN', 'волчий_NOUN', 'койот_NOUN', 'волчонок_NOUN']
Домен: еда
хлеб_NOUN -> ['хлеба_NOUN', 'хлебец_NOUN', 'хлебушек_NOUN', 'хлебный_ADJ', 'хлеб_PROPN', 'хлебль_NOUN', 'пшено_NOUN', 'хлбъ_PROPN', 'мучица_

## Обучение Word2Vec и сравнение ближайших соседей с готовыми эмбеддингами

### Исходный датасет

В качестве исходного датасета используется [Kostya165/ru_emotion_dvach](https://huggingface.co/datasets/Kostya165/ru_emotion_dvach) для задач классификации эмоционального интента.

In [46]:
ru_emotion_dvach_ds = load_dataset("Kostya165/ru_emotion_dvach")
texts = ru_emotion_dvach_ds['train']['text']
labels = ru_emotion_dvach_ds['train']['label']

print("Количество документов:", len(texts))
print("Количество категорий", len(Counter(labels)))

Количество документов: 59061
Количество категорий 5


Сделаем простую токенизацию исходного корпуса.

In [47]:
def get_tokens(text: str) -> List[str]:
    text = text.lower()
    tokens = re.findall(r"[а-яё]+(?:-[а-яё]+)?", text)

    return tokens


texts_tokenized = [get_tokens(t) for t in texts if t]

print("Примеры токенов:", texts_tokenized[0][:30])

Примеры токенов: ['я', 'боюсь', 'что', 'из-за', 'контроля', 'сварных', 'соединений', 'я', 'могу', 'пропустить', 'что-то', 'важное', 'и', 'тогда', 'будут', 'большие', 'проблемы']


### Обучение Word2Vec

Обучим Word2Vec на полученных токенах.

In [48]:
w2v = Word2Vec(
    sentences=texts_tokenized,
    vector_size=100,
    window=5,
    min_count=3,
    sg=1,
    negative=10,
    epochs=10,
    workers=4,
)

wv = w2v.wv

print("Словарь:", len(wv))
print("Размерность:", wv.vector_size)

Словарь: 39092
Размерность: 100


Сравним ближайшие слова из обученного W2V с готовой моделью. Из полученных результатов можно заметить:

1. В список ближайших слов к слову "волк" есть прилагательное "тамбовский" (фразеологизм "тамбовский волк");
2. Рядом со словом "хлеб" присутствуют два слова – "литл" и "биг". Предположительно, это связанно с тем, что "хлеб" и "литл биг" – музыкальные группы и они были упомянуты вместе;
3. Соседнее со словом "мороз" – "снегурочка-дед" (Дед Мороз и Снегурочка).

In [49]:
for domain in words:
    print("Домен:", domain)

    for word in words[domain]:
        print_neighbors(model, word + "_NOUN")
        print_neighbors(wv, word)

    print()

Домен: животные
кошка_NOUN -> ['кот_NOUN', 'котенок_NOUN', 'котенка_NOUN', 'котенок_VERB', 'собака_NOUN', 'щенок_NOUN', 'собачка_NOUN', 'кошка_PROPN', 'дворняжка_NOUN', 'крыса_NOUN']
кошка -> ['клёвая', 'милая', 'ворона', 'жирная', 'лодочка', 'независимая', 'родственница', 'умерла', 'пиратская', 'крупная']
собака_NOUN -> ['пес_NOUN', 'дворняжка_NOUN', 'дворняга_NOUN', 'легавый_ADJ', 'кошка_NOUN', 'собачонка_NOUN', 'овчарка_NOUN', 'щенок_NOUN', 'гончий_ADJ', 'собачка_NOUN']
собака -> ['стайная', 'кусала', 'поворачиваюсь', 'роялю', 'лежу', 'летает', 'болен', 'врет', 'хуета', 'интеллктуалам']
лошадь_NOUN -> ['конь_NOUN', 'лошадь_PROPN', 'жеребец_NOUN', 'лошадь_ADV', 'кобыла_NOUN', 'повозка_NOUN', 'лошадка_NOUN', 'телега_NOUN', 'иноходец_NOUN', 'мул_NOUN']
лошадь -> ['разбегу', 'стожок', 'интерактивными', 'слепая', 'закрепленной', 'фазу', 'веревкой', 'брэда', 'питта', 'ёбнет']
медведь_NOUN -> ['медведь_PROPN', 'медведь_VERB', 'зверь_NOUN', 'волк_NOUN', 'медведица_NOUN', 'барсук_NOUN', 'тиг

## Построение индекса по датасету

Для данного задания импортируем ещё один датасет ([IlyaGusev/gazeta](https://huggingface.co/datasets/IlyaGusev/gazeta)).

In [50]:
gazeta_ds = load_dataset("IlyaGusev/gazeta")
texts = gazeta_ds['train']['text']

print("Количество документов:", len(texts))
print("Примеры документов:")
texts[:3]

Количество документов: 60964
Примеры документов:


['Сегодня транспортный налог начисляется в зависимости от мощности автомобиля, причем цена для «сильных» машин выше, чем для малолитражек. Также ставку налога могут корректировать региональные власти: согласно Налоговому кодексу, базовый тариф, установленный правительством, может быть уменьшен в пять раз или увеличен до 10 раз. Сборы идут в региональные бюджеты, откуда растекаются на общие нужды. Транспортный налог — один из основных источников бюджетных доходов — предлагается направить исключительно на дорожные фонды. Так, автомобилисты будут понимать, за что они платят, а дорожники будут иметь гарантированный доход. Кроме налога дорожные фонды будут пополняться за счет бюджетных средств и проезда по платным дорогам. Более того, транспортный налог предлагается завуалировать в акцизы на бензин. Привычную и раздражающую систему ежегодной оплаты квитанции предлагается изменить, включив налог в стоимость топлива. Минэкономразвития говорит об удвоении акцизы, которая сегодня составляет 3,3

Обучим Word2Vec на новом датасете.

In [51]:
texts_tokenized = [get_tokens(t) for t in texts[:4000] if t]

w2v = Word2Vec(
    sentences=texts_tokenized,
    vector_size=100,
    window=5,
    min_count=3,
    sg=1,
    negative=10,
    epochs=10,
    workers=4,
)

wv = w2v.wv

print("Словарь:", len(wv))
print("Размерность:", wv.vector_size)

Словарь: 59703
Размерность: 100


Построим эмбеддинг документа.

In [52]:
def doc_vector_mean(tokens, wv, dim):
    vecs = [wv[w] for w in tokens if w in wv]

    if not vecs:
        return np.zeros(dim, dtype=np.float32)

    return np.mean(vecs, axis=0).astype(np.float32)


def doc_vector_tfidf(tokens, wv, tfidf_row, vocab, dim):
    weights = {}

    for w in tokens:
        j = vocab.get(w, None)

        if j is not None:
            weights[w] = tfidf_row[0, j]

    num = np.zeros(dim, dtype=np.float32)
    den = 0.0

    for w, a in weights.items():
        if w in wv and a > 0:
            num += (a * wv[w]).astype(np.float32)
            den += float(a)

    if den == 0.0:
        return doc_vector_mean(tokens, wv, dim)

    return (num / den).astype(np.float32)


docs = texts[:1000]

docs_tok = [get_tokens(t) for t in docs]
dim = wv.vector_size

vectorizer = TfidfVectorizer(
    tokenizer=get_tokens,
    lowercase=True,
    min_df=2,
    max_df=0.9,
)
tfidf = vectorizer.fit_transform(docs)
vocab = vectorizer.vocabulary_

D_mean = np.vstack([doc_vector_mean(t, wv, dim) for t in docs_tok])
D_tfidf = np.vstack([
    doc_vector_tfidf(docs_tok[i], wv, tfidf[i], vocab, dim)
    for i in range(len(docs_tok))
])

print("D_mean_shape:", D_mean.shape, "D_tfidf shape:", D_tfidf.shape)

D_mean_n = D_mean / (np.linalg.norm(D_mean, axis=1, keepdims=True) + 1e-9)
D_tfidf_n = D_tfidf / (np.linalg.norm(D_tfidf, axis=1, keepdims=True) + 1e-9)

print("D_mean_n shape:", D_mean_n.shape, "D_tfidf_n shape:", D_tfidf_n.shape)

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


D_mean_shape: (1000, 100) D_tfidf shape: (1000, 100)
D_mean_n shape: (1000, 100) D_tfidf_n shape: (1000, 100)


Далее сделаем 5 запросов. Можно заметить следующее:

1. Dense и FAISS дают практически идентичные результаты
2. TF-IDF лучше справляется с точным лексическим совпадением (например, по запросу «запуск ракеты космос МКС» TF-IDF сразу выдаёт релевантную статью о запуске «Союза» к МКС с высоким скором (0.29), а «шум» от нерелевантных документов минимален). Когда в запросе есть редкие, специфичные термины (МКС, ДТП, курс доллара), TF-IDF работает лучше.

3. Dense-модели лучше улавливают семантику, но иногда «размывают» релевантность  
По запросу «футбол чемпионат мира сборная» dense-модель в топ подтягивает статьи про волейбольную сборную — семантически близко (командный спорт, сборная, международные соревнования), но тематически это промах.

4. Dense-модели выигрывают на абстрактных и многословных запросах

5. Скоры dense-моделей значительно выше и стабильнее

Для данного корпуса оптимальным подходом был бы гибридный поиск – комбинация TF-IDF (для точного лексического matching) и dense retrieval (для семантического обобщения). TF-IDF надёжнее при наличии уникальных ключевых слов, dense — при нечётких или обобщённых формулировках. FAISS не добавляет качества относительно «наивного» dense cosine, но обеспечивает масштабируемость на больших корпусах.1

In [53]:
def cosine_topk_pre_norm(query_vec, Dnrm, k):
    q = query_vec.astype(np.float32)
    q = q / (np.linalg.norm(q) + 1e-9)
    scores = Dnrm @ q
    idx = np.argsort(-scores)[:k]

    return idx, scores[idx]


queries = [
    "выборы президента и результаты голосования",
    "курс доллара и экономический кризис",
    "футбол чемпионат мира сборная",
    "запуск ракеты космос МКС",
    "авария ДТП пострадавшие",
]
K = 5

for q in queries:
    print("=" * 90)
    print("Запрос:", q)

    q_tfidf = vectorizer.transform([q])
    scores_tfidf = (tfidf @ q_tfidf.T).toarray().ravel()
    idx_tfidf = np.argsort(-scores_tfidf)[:K]

    print("\nTF-IDF топ-k:")
    for r, i in enumerate(idx_tfidf, 1):
        snippet = re.sub(r"\s+", " ", docs[i])[:200]
        print(f"{r:>2}. score={scores_tfidf[i]:.4f} | {snippet}...")

    tok = get_tokens(q)
    qv = doc_vector_tfidf(tok, wv, q_tfidf, vocab, dim)
    idx_dense, sc_dense = cosine_topk_pre_norm(qv, D_tfidf_n, k=K)

    print("\nDense (cosine) топ-k:")
    for r, (i, s) in enumerate(zip(idx_dense, sc_dense), 1):
        snippet = re.sub(r"\s+", " ", docs[i])[:200]
        print(f"{r:>2}. score={float(s):.4f} | {snippet}...")

    D = D_tfidf_n.astype(np.float32)
    index = faiss.IndexFlatIP(D.shape[1])
    index.add(D)
    q_tfidf = vectorizer.transform([q])
    qv = doc_vector_tfidf(tok, wv, q_tfidf, vocab, dim).astype(np.float32)
    qv /= (np.linalg.norm(qv) + 1e-9)
    scores, idx = index.search(qv.reshape(1, -1), K)
    idx = idx[0]
    scores = scores[0]

    print("\nCosine sim (FAISS) топ-k:")
    for r, (i, s) in enumerate(zip(idx, scores), 1):
        snippet = re.sub(r"\s+", " ", docs[int(i)])[:200]
        print(f"{r:>2}. score={float(s):.3f} | {snippet}...")

    print()

Запрос: выборы президента и результаты голосования

TF-IDF топ-k:
 1. score=0.1431 | В пятницу в подмосковном Новогорске состоялась отчетно-выборная конференция Федерации фигурного катания на коньках России (ФФККР). Действующий президент Федерации, как и обещал, свою кандидатуру на вы...
 2. score=0.1371 | На выборах президента Польши победу одержал председатель польского парламента Бронислав Коморовский , который исполнял обязанности главы страны после трагической гибели Леха Качиньского в Смоленске в ...
 3. score=0.1044 | В выходные украинский парламент демонстрировал завидное трудолюбие. Рада заседала до 9 вечера субботы и была готова продолжить работу в воскресенье, чтобы выполнить задачу, поставленную руководством с...
 4. score=0.1032 | Испанская «Барселона» в ночь на понедельник выбирала нового президента сроком на четыре года. Вопреки воле действующего главы клуба Жоана Лапорты, президентом команды стал не его официальный преемник ...
 5. score=0.0921 | В пятницу парламент Каз